# Pydantic AI together with Pydantic Models
#### Construct pydantic model from text input

In [6]:
# Agent: the object that sends your questions to the AI model and returns the answer
from pydantic_ai import Agent

# OpenAIChatModel: a wrapper that talks to any service using OpenAI's chat format
from pydantic_ai.models.openai import OpenAIChatModel

# OpenAIProvider: holds the connection details (where to connect and how to log in)
from pydantic_ai.providers.openai import OpenAIProvider

import os  # lets us read environment variables

from dotenv import load_dotenv  # reads key=value pairs from a .env file

# Load the variables from your .env file (e.g. NVIDIA_API=...) into the environment.
# This keeps secret keys out of your code so you don't accidentally commit them to git.
load_dotenv()

# Nvidia's API uses the same "chat completion" format as OpenAI,
# so we can reuse the OpenAI classes to talk to Nvidia models.
model = OpenAIChatModel(
    "nvidia/nemotron-3.5-lightning-30b-a3b",
    provider=OpenAIProvider(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=os.environ["NVIDIA_API"],
    ),
)


In [7]:
# from Model import nvidia_model
from pydantic_ai import Agent

employee_simulator = Agent(
    model = nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
    instructions = "You are an expert at simulating employment in Sweden. Simulate it in this format."
    """
        name: "Bertil",
        age: 40,
        salary: 35000,
        role: "Data Engineer",
        area: "Stockholm"
    """
)

In [8]:
# this is quite fast model that also has tool use
agent = Agent(model=model)

result = await agent.run("Give me an IT employee working in Sweden, shortly.", timeout = 60)
result

                 pydantic-ai v2.62.0 • Python 3.13.7

      / \        agent: agent • model: openai:nvidia/nemotron-3.5-lightning-30b-a3b • tools: 0
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


AgentRunResult(output="Johan, 31, is a DevOps engineer in Stockholm, specializing in CI/CD and cloud automation, enjoying Sweden's flexible work culture and strong emphasis on work-life balance.")

In [9]:
print(result.output)

Johan, 31, is a DevOps engineer in Stockholm, specializing in CI/CD and cloud automation, enjoying Sweden's flexible work culture and strong emphasis on work-life balance.


## Why `BaseModel`?
- When you write class EmployeeModel(BaseModel), your class inherits Pydantic’s abilities. Without it, EmployeeModel would be a plain Python class that checks nothing.

- With it, Pydantic automatically:
    - validates types: age: int rejects "hello".
    - converts when safe: "34" becomes 34.
    - raises a clear error when the data doesn’t match.
- For PydanticAI, this matters because the LLM returns text. BaseModel turns that text into a checked Python object, and the model’s field names and types tell the LLM what format to answer in.

## Why `Field`?
- A bare type like salary: int only says “a whole number”. Field lets you say more:
    - `gt=30_000, lt=50_000:` value rules (greater than, less than).
    - `description="...":` a hint for the LLM about what to put in this field.
- You only need Field for fields that need extra rules or hints. Plain fields like `name:` str can stay without it.

- Analogy for a data engineer: `BaseModel` is like defining a table schema `(columns and types)`
- `Field` is like adding constraints and column comments `(CHECK salary > 30000, COMMENT '...')`.


In [10]:
# BaseModel: the Pydantic class we inherit from to define the "shape" of our data.
# Field: lets us add extra rules and descriptions to a single field.
from pydantic import BaseModel, Field

class EmployeeModel(BaseModel):
    """Describes what one employee record should look like.

    Pydantic checks the data against this blueprint. If the LLM returns
    something that doesn't fit (e.g. age as text, or a salary that is too
    high), Pydantic raises an error instead of silently accepting it.
    """
    name: str
    age: int
    salary: int = Field(description="salary should be between 30k and 50k, the more senior the position, the higher salary", gt=30_000, lt=50_000)
    position: str

    # The description is sent to the LLM as a hint, so it knows what to generate.


# `await` waits for the LLM to answer without freezing the notebook.
# (It works at the top level because Jupyter notebooks allow it.)
# - The text is the prompt we send to the LLM.
# - output_type=EmployeeModel tells the agent to reply in the shape of
#   EmployeeModel, not as free text. Pydantic then validates the reply.
result = await agent.run(
    "Give me an IT employee working in sweden", output_type=EmployeeModel
)
result

AgentRunResult(output=EmployeeModel(name='Anna Lindberg', age=30, salary=45000, position='Software Engineer'))

In [11]:
result.output

EmployeeModel(name='Anna Lindberg', age=30, salary=45000, position='Software Engineer')

In [16]:
type(result.output)

list

In [12]:
result.output.name, result.output.age, result.output.salary

('Anna Lindberg', 30, 45000)

In [13]:
result.output.model_dump()

{'name': 'Anna Lindberg',
 'age': 30,
 'salary': 45000,
 'position': 'Software Engineer'}

In [14]:
result = await agent.run(
    "Give me ten employees in AI and data engineering fields, so the roles can vary, salary must be between 30000 and 50000",
    output_type=list[EmployeeModel],
)
result

AgentRunResult(output=[EmployeeModel(name='Alice Johnson', age=28, salary=45000, position='Machine Learning Engineer'), EmployeeModel(name='Bob Smith', age=34, salary=48000, position='Data Scientist'), EmployeeModel(name='Cara Lee', age=30, salary=42000, position='AI Researcher'), EmployeeModel(name='Dan King', age=35, salary=46000, position='NLP Engineer'), EmployeeModel(name='Eve Martinez', age=29, salary=44000, position='Computer Vision Engineer'), EmployeeModel(name='Frank Adams', age=40, salary=47000, position='Data Engineer'), EmployeeModel(name='Grace White', age=33, salary=43000, position='Data Platform Engineer'), EmployeeModel(name='Hank Black', age=38, salary=41000, position='ETL Developer'), EmployeeModel(name='Ivy Green', age=31, salary=40000, position='Database Administrator'), EmployeeModel(name='Jack Brown', age=36, salary=49000, position='Big Data Engineer')])

In [15]:
result.output

[EmployeeModel(name='Alice Johnson', age=28, salary=45000, position='Machine Learning Engineer'),
 EmployeeModel(name='Bob Smith', age=34, salary=48000, position='Data Scientist'),
 EmployeeModel(name='Cara Lee', age=30, salary=42000, position='AI Researcher'),
 EmployeeModel(name='Dan King', age=35, salary=46000, position='NLP Engineer'),
 EmployeeModel(name='Eve Martinez', age=29, salary=44000, position='Computer Vision Engineer'),
 EmployeeModel(name='Frank Adams', age=40, salary=47000, position='Data Engineer'),
 EmployeeModel(name='Grace White', age=33, salary=43000, position='Data Platform Engineer'),
 EmployeeModel(name='Hank Black', age=38, salary=41000, position='ETL Developer'),
 EmployeeModel(name='Ivy Green', age=31, salary=40000, position='Database Administrator'),
 EmployeeModel(name='Jack Brown', age=36, salary=49000, position='Big Data Engineer')]

## Task: Calculate mean salary

In [17]:
from statistics import mean

mean_salary = mean(employee.salary for employee in result.output)

print(mean_salary)

44500


## Task: Convert to Pandas DataFrame

In [18]:
import pandas as pd
pd.DataFrame([employee.model_dump() for employee in result.output])

,name,age,salary,position
0,Alice Johnson,28,45000,Machine Learning Engineer
1,Bob Smith,34,48000,Data Scientist
2,Cara Lee,30,42000,AI Researcher
3,Dan King,35,46000,NLP Engineer
4,Eve Martinez,29,44000,Computer Vision Engineer
5,Frank Adams,40,47000,Data Engineer
6,Grace White,33,43000,Data Platform Engineer
7,Hank Black,38,41000,ETL Developer
8,Ivy Green,31,40000,Database Administrator
9,Jack Brown,36,49000,Big Data Engineer


# CV model - a more complex and nested model

In [15]:
class ExperienceModel(BaseModel):
    title: str
    company: str
    description: str
    start_year: int
    end_year: int


class EducationModel(BaseModel):
    title: str
    education_area: str
    school: str
    description: str
    start_year: int
    end_year: int


class CvModel(BaseModel):
    name: str
    age: int
    experiences: list[ExperienceModel]
    educations: list[EducationModel]


result = await agent.run(
    "Create 5 fake persons that is applying for a data engineering job. They should have 2-5 previous job experiences",
    output_type=list[CvModel],
)

In [16]:
result.output[-1]

CvModel(name='Hank Rogers', age=33, experiences=[ExperienceModel(title='Database Administrator', company='SysTech', description='Managed relational databases, performed tuning, and ensured backup/recovery strategies.', start_year=2018, end_year=2019), ExperienceModel(title='Data Engineer', company='DataMind', description='Built ETL pipelines using Python and Apache Airflow, integrating with Power BI for reporting.', start_year=2019, end_year=2021), ExperienceModel(title='Senior Data Engineer', company='InsightAI', description='Designed machine learning data pipelines using TensorFlow and Apache Spark, supporting predictive models.', start_year=2021, end_year=2023), ExperienceModel(title='Lead Data Engineer', company='DataForge', description='Oversaw the implementation of a real-time data fabric using Apache Pulsar and Flink.', start_year=2023, end_year=2024)], educations=[EducationModel(title='M.S. in Computer Science', education_area='Computer Science', school='Carnegie Mellon Univers

In [17]:
result.output[-1].name

'Hank Rogers'

In [18]:
result.output[-1].experiences[0].title, result.output[-1].experiences[1].title

('Database Administrator', 'Data Engineer')

# (optional) Postprocessing - load into duckdb and unnesting
- This part is optional, but a way to unnest the data and store it could be to use dlt to load the data into duckdb, followed by joining and unnesting.

- Other approach could be to store into nosql such as mongodb.

In [ ]:
import dlt

pipeline = dlt.pipeline(
    pipeline_name="cv_duckdb",
    destination=dlt.destinations.duckdb("cv.duckdb"),
    dataset_name="staging",
)

info = pipeline.run(
    data=[result.output], loader_file_format="jsonl", table_name="cv_entries"
)

print(info)

In [ ]:
import duckdb 

with duckdb.connect("cv.duckdb") as conn:
    desc = conn.sql("desc;").df()
    cv_entries = conn.sql("FROM staging.cv_entries").df()
    educations = conn.sql("FROM staging.cv_entries__educations").df()
    experiences = conn.sql("FROM staging.cv_entries__experiences").df()

desc

In [ ]:
cv_entries

In [ ]:
educations

In [ ]:
experiences

In [ ]:
duckdb.sql("""
    SELECT 
        cv.name, 
        cv.age, 
        ex.company,
        ex.description AS experience_description,
        ex.start_year AS experience_start_year,
        ex.end_year AS experience_end_year,
        e.title,
        e.education_area,
        e.school,
        e.start_year AS education_start_year,
        e.end_year AS education_end_year
    FROM cv_entries cv
    LEFT JOIN educations e ON cv._dlt_id = e._dlt_parent_id
    LEFT JOIN experiences ex ON cv._dlt_id = ex._dlt_parent_id
    

""").df()